# Jarvis — Hosted Brain + Kaggle Dual-GPU Voice: RUN IN ORDER

**Version: temporary free hosted Groq/Gemini brain (Nov 2026).** This replaces the Qwen3-0.6B reply engine that was parroting your words, and frees GPU0 memory for higher-accuracy Whisper Medium. OmniVoice keeps your cloned English/Tamil voice on GPU1.

**Before Run All:** Stop the Windows client, stop old Kaggle uvicorn/tunnel processes, restart your Kaggle session, attach the existing PRIVATE voice dataset, and enable T4 ×2 and Internet.

**Secrets:** Add Kaggle `VOICE_API_TOKEN` (rotated if previously exposed) and **`GROQ_API_KEY`** (or `GEMINI_API_KEY`). Groq's current recommended fast model is `openai/gpt-oss-20b` (the old Llama model IDs were retired). Gemini's `gemini-3.5-flash-lite` offers a free tier; both have usage limits.

**Run every cell top to bottom**, without mixing in your old local-Qwen settings. Stop immediately on an error. This improves latency but is *still modular*, not native human-speed audio-to-audio full-duplex.

## 1. Update project code (works for both an existing repo and a fresh notebook)

In [ ]:
import os, subprocess, sys, time
from pathlib import Path

REPO = Path("/kaggle/working/jarvis-realtime-voice")
GIT_URL = "https://github.com/Godf-Cyb3r-1/jarvis-realtime-voice.git"

if (REPO / ".git").is_dir():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
else:
    if REPO.exists() and any(REPO.iterdir()):
        raise RuntimeError("The project folder exists without .git. Rename/remove it manually, then rerun.")
    subprocess.run(["git", "clone", GIT_URL, str(REPO)], check=True)

os.chdir(REPO)
for name in ["duplex_server.py", "duplex_client.py", "gpu_guard.py",
             "tts_engine.py", "local_llm.py", "requirements-duplex-server.txt"]:
    assert (REPO / name).is_file(), f"Missing source file: {name}"
print("PASS: updated Jarvis repo at", REPO)

## 2. Check installed packages — only install if something is missing

Your uploaded notebook already successfully imported OmniVoice, Whisper, ONNX, and Transformers after installation. This cell **skips pip** when they are present. If it installs anything, it intentionally stops; **restart the Python session and rerun from Step 1** so CUDA and PyTorch see a consistent environment.

In [ ]:
import importlib.util
needed = {
    "torch": "torch", "ctranslate2": "ctranslate2", "faster_whisper": "faster_whisper",
    "omnivoice": "omnivoice", "transformers": "transformers",
    "fastapi": "fastapi", "uvicorn": "uvicorn", "soundfile": "soundfile",
    "httpx": "httpx", "scipy": "scipy", "requests": "requests",
    "onnxruntime": "onnxruntime"
}
missing = [name for name, mod in needed.items() if importlib.util.find_spec(mod) is None]
if missing:
    print("Missing packages:", missing)
    if any(name != "omnivoice" for name in missing):
        subprocess.run([sys.executable, "-m", "pip", "install", "-r",
                        str(REPO / "requirements-duplex-server.txt")], check=True)
    if "omnivoice" in missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "omnivoice"], check=True)
    raise RuntimeError("Packages installed. RESTART the Python session, then run this notebook again from Step 1.")
print("PASS: all required dependencies already installed; no reinstall was done")

## 3. Confirm BOTH frameworks can see GPU 0 and GPU 1

In [ ]:
import torch, ctranslate2, transformers, omnivoice, faster_whisper
print("PyTorch:", torch.__version__, "| Transformers:", transformers.__version__)
print("Torch CUDA GPUs:", torch.cuda.device_count())
print("CTranslate2 CUDA GPUs:", ctranslate2.get_cuda_device_count())
assert torch.cuda.is_available(), "Enable Kaggle GPU accelerator"
assert torch.cuda.device_count() >= 2, "This notebook requires T4 ×2; don't assign GPU 1 on a single GPU"
assert ctranslate2.get_cuda_device_count() >= 2, "CTranslate2 cannot see GPU 1; stop before loading models"
for i in range(2):
    free, total = torch.cuda.mem_get_info(i)
    print(f"GPU {i}: {torch.cuda.get_device_name(i)} | {free / 2**30:.2f} / {total / 2**30:.2f} GiB free")
print("PASS: dual-GPU device assignment is available")

## 4. Prepare your ALREADY-ATTACHED private voices (no ZIP needed)

This uses the raw `jarvis_en/new-ref.mp3` and `jarvis_ta/ref.mp3` paths that were present in your uploaded notebook. The reference transcripts are copied unmodified.

In [ ]:
import shutil
import soundfile as sf

dataset_candidates = []
for en_file in Path("/kaggle/input").rglob("new-ref.mp3"):
    dataset = en_file.parent.parent
    required = [
        dataset / "jarvis_en/new-ref.mp3", dataset / "jarvis_en/ref.txt",
        dataset / "jarvis_ta/ref.mp3", dataset / "jarvis_ta/ref.txt"
    ]
    if en_file.parent.name == "jarvis_en" and all(p.is_file() for p in required):
        dataset_candidates.append(dataset)
assert dataset_candidates, ("Reference dataset not attached. Check Kaggle Input for your private "
                            "jarvis-voice-references dataset.")
DATASET = dataset_candidates[0]
VOICE_DIR = REPO / "voices"
VOICE_DIR.mkdir(parents=True, exist_ok=True)
for lang, audio_rel, text_rel in [
    ("en", "jarvis_en/new-ref.mp3", "jarvis_en/ref.txt"),
    ("ta", "jarvis_ta/ref.mp3", "jarvis_ta/ref.txt")
]:
    source = VOICE_DIR / f"{lang}.mp3"
    shutil.copy2(DATASET / audio_rel, source)
    shutil.copy2(DATASET / text_rel, VOICE_DIR / f"{lang}.txt")
    subprocess.run(["ffmpeg", "-y", "-v", "error", "-i", str(source),
                    "-ac", "1", "-ar", "24000", str(VOICE_DIR / f"{lang}.wav")], check=True)
    info = sf.info(VOICE_DIR / f"{lang}.wav")
    assert info.samplerate == 24000 and info.frames > 0
    assert (VOICE_DIR / f"{lang}.txt").read_text(encoding="utf-8").strip()
    print(lang.upper(), "voice OK:", round(info.duration, 1), "seconds @", info.samplerate, "Hz")
print("PASS: your English and Tamil voice references are ready (still private)")

## 5. Load Groq or Gemini key and set low-latency voice configuration **before** starting the server

Create your API key at [Groq Console](https://console.groq.com/keys) for the default `HOSTED_BACKEND="groq"`. Alternatively set it to `"gemini"` and create a Gemini key at [Google AI Studio](https://aistudio.google.com/apikey). Add the corresponding key to Kaggle Secrets and enable notebook access.

This frees GPU0 of the local Qwen model, lets us use Whisper Medium for better bilingual recognition, and reduces the OmniVoice diffusion-step count to 8 as a *speed/quality trade-off* (if the voice sounds poor, increase to 12).

In [ ]:
import site
from kaggle_secrets import UserSecretsClient
secrets_client=UserSecretsClient()

# Choose one temporary free-tier provider: "groq" or "gemini"
HOSTED_BACKEND="groq"

try:
    private_token=secrets_client.get_secret("VOICE_API_TOKEN")
    provider_key_name="GROQ_API_KEY" if HOSTED_BACKEND=="groq" else "GEMINI_API_KEY"
    provider_secret=secrets_client.get_secret(provider_key_name)
except Exception as exc:
    raise RuntimeError("Add Kaggle Secrets VOICE_API_TOKEN and "+provider_key_name+" before continuing") from exc

assert len(private_token)>=24, "Create a NEW private connection token (at least 24 characters)"
assert provider_secret.strip(), "Missing free hosted-model API key"
os.environ["VOICE_API_TOKEN"]=private_token
os.environ[provider_key_name]=provider_secret
del private_token, provider_secret

os.environ.update({
    "LLM_BACKEND": HOSTED_BACKEND,
    "GROQ_MODEL": "openai/gpt-oss-20b",
    "GEMINI_MODEL": "gemini-3.5-flash-lite",
    "LLM_MAX_TOKENS": "140",
    "ASR_DEVICE": "cuda",
    "ASR_GPU_INDEX": "0",
    "ASR_MODEL": "medium",
    "ASR_COMPUTE": "int8_float16",
    "ASR_BEAM_SIZE": "3",
    "ASR_LANGUAGE": "auto",
    "ASR_ALLOWED_LANGUAGES": "en,ta",
    "ASR_HOTWORDS": "Jarvis",
    "ASR_MIN_RMS": "0.003",
    "ASR_MIN_AVG_LOGPROB": "-1.25",
    "PARTIAL_ASR_ENABLED": "0",
    "SMART_TURN_ENABLED": "0",
    "ENDPOINT_FALLBACK_MS": "850",
    "TTS_PROVIDER": "omnivoice",
    "TTS_GPU": "1",
    "VOICE_DIR": str(VOICE_DIR),
    "OMNIVOICE_STEPS": "8",
    "ALLOW_BACKCHANNEL": "0",
    "PYTHONUNBUFFERED": "1",
})
# Set CUDA library paths BEFORE the server starts, not after.
libs=[]
for site_dir in site.getsitepackages():
    for name in ("cublas","cudnn","cuda_runtime"):
        path=Path(site_dir)/"nvidia"/name/"lib"
        if path.is_dir():libs.append(str(path))
old_path=os.environ.get("LD_LIBRARY_PATH","")
os.environ["LD_LIBRARY_PATH"]=":".join(dict.fromkeys(libs+([old_path] if old_path else [])))
print("PASS: Groq/Gemini + Whisper + OmniVoice configured; NO API keys printed")
print("Brain:", HOSTED_BACKEND, "hosted (no local Qwen GPU)")
print("GPU 0: Whisper medium | GPU 1: OmniVoice")


## 6. Validate source and run offline tests (including hosted LLM and ASR tests)

In [ ]:
subprocess.run([sys.executable, "-m", "compileall", "-q",
    "gpu_guard.py", "duplex_server.py", "duplex_client.py",
    "tts_engine.py", "local_llm.py", "smart_turn_adapter.py",
    "hosted_llm.py", "asr_quality.py"], check=True)
subprocess.run([sys.executable, "-m", "unittest", "discover",
    "-s", "tests", "-p", "test_*.py", "-v"], check=True)
print("PASS: syntax, provider and ASR policy tests")

## 7. Start ONE server only

Do **not** run `OmniVoiceEngine()` directly in another Kaggle cell: that can load a duplicate GPU model. This cell checks whether an older process still owns port 8091 and refuses to silently use the wrong server.

In [ ]:
import socket
import requests

if "server" in globals() and server is not None and server.poll() is None:
    server.terminate()
    server.wait(timeout=20)
    time.sleep(2)

with socket.socket() as sock:
    port_busy = sock.connect_ex(("127.0.0.1", 8091)) == 0
if port_busy:
    raise RuntimeError("Port 8091 is already in use by an OLD server. Stop it or restart Kaggle session.")

SERVER_LOG = Path("/kaggle/working/jarvis_server.log")
server_log_handle = SERVER_LOG.open("w")
server = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "duplex_server:app",
     "--host", "127.0.0.1", "--port", "8091"],
    cwd=REPO, env=os.environ.copy(),
    stdout=server_log_handle, stderr=subprocess.STDOUT
)
health = None
for _ in range(180):
    if server.poll() is not None:
        break
    try:
        response = requests.get("http://127.0.0.1:8091/health", timeout=2)
        if response.ok:
            health = response.json()
            break
    except requests.RequestException:
        pass
    time.sleep(1)
if not health:
    print(SERVER_LOG.read_text(errors="replace")[-5000:])
    raise RuntimeError("Jarvis server did not start. See log above.")

print("SERVER:", {k: health.get(k) for k in
    ("ready", "asr_model", "asr_device", "asr_gpu_index", "llm_backend",
     "tts_provider", "smart_turn")})
assert health.get("asr_gpu_index") == 0, "Server did not use GPU 0 for Whisper"
assert health.get("tts_provider") == "omnivoice"
print("PASS: one correct Jarvis server is running")

## 8. Verify the hosted brain responds meaningfully (no local Qwen GPU usage)

The provider must return an actual conversational answer, not merely repeat the transcript. If the provider returns HTTP 401/429, check the API key or free-tier quota and stop here. Do not start Windows until this passes.

In [ ]:
HEADERS = {"Authorization": "Bearer " + os.environ["VOICE_API_TOKEN"]}
question="Hello Jarvis, what can we talk about together?"
try:
    r=requests.post(
        "http://127.0.0.1:8091/debug/llm", headers=HEADERS,
        json={"text":question}, timeout=180
    )
    print("Hosted LLM HTTP:", r.status_code)
    print("Provider reply:", r.text[:900])
    if not r.ok or not r.json().get("reply","").strip():
        print("SERVER LOG:", SERVER_LOG.read_text(errors="replace")[-2500:])
        raise RuntimeError("Hosted LLM did not return a reply; check Kaggle secret and provider quota.")
except requests.RequestException:
    print("SERVER LOG:", SERVER_LOG.read_text(errors="replace")[-2500:])
    raise
print("PASS: hosted conversational LLM responds (no local Qwen needed)")

## 9. Check OmniVoice English and Tamil inside ONE server

This tests voice cloning with the correct sample rate. The **full-text bug has been fixed** in `/debug/tts`: instead of silently discarding text after 200 characters, the server splits long input into speech chunks and joins the audio. It reports the original input-character count and number of generated chunks in response headers. The next optional Step 9B checks your actual long sample.

In [ ]:
from IPython.display import Audio, display
for lang, text in [
    ("en", "Hey, I'm Jarvis. Nice to hear from you."),
    ("ta", "வணக்கம்! நான் ஜார்விஸ். எப்படி இருக்கீங்க?")
]:
    try:
        r = requests.post("http://127.0.0.1:8091/debug/tts",
            headers=HEADERS, json={"text": text, "language": lang}, timeout=300)
    except requests.RequestException:
        print("SERVER LOG:", SERVER_LOG.read_text(errors="replace")[-3000:])
        raise
    print(f"OmniVoice {lang} HTTP:", r.status_code)
    print("Requested characters:",len(text),
          "| Received:",r.headers.get("X-TTS-Input-Characters"),
          "| Speech chunks:",r.headers.get("X-TTS-Chunks"))
    if not r.ok:
        print("Response:", r.text[:1000])
        print("SERVER LOG:", SERVER_LOG.read_text(errors="replace")[-3000:])
        raise RuntimeError(f"OmniVoice {lang} failed; do not continue yet.")
    target = Path("/kaggle/working") / f"jarvis_checked_{lang}.wav"
    target.write_bytes(r.content)
    info = sf.info(target)
    assert info.frames > 0 and info.samplerate == 16000
    print("Voice audio:", round(info.duration, 2), "seconds, 16000 Hz")
    display(Audio(filename=str(target)))
print("PASS: both English and Tamil speech synthesis work in the SERVER")

## 9B. Long-text completeness test — your paragraph (optional, slower)The code below uses your long English and Tamil inputs. Set `RUN_LONG_TTS_TEST=True` and run only this cell after Step 9. It checks that the endpoint **did not discard input characters**, generates all speech segments, and plays one combined WAV. **Listen for the last English words**, not just the reported duration. Long-form TTS will take longer than a short single response. Expressive aliases `[chuckle]` and `[short pause]` are mapped to supported equivalents.

In [ ]:
RUN_LONG_TTS_TEST = False  # Change to True to generate your full long samples
if RUN_LONG_TTS_TEST:
    tests = [
        ("en", "[laughter] How have you been holding up with everything lately? I know the past few weeks have been super busy on your end... [sigh] Honestly, I feel like this whole month just flew right past us! Whenever you finally catch a free moment to breathe, let's grab a coffee or hop on a quick call so you can tell me all about how the new project is going. [laughter] No rush at all though—just reply whenever you get a chance!"),
        ("ta", "[chuckle] சரி, புரிஞ்சிது! நான் இப்போ உள்ள இருக்க தகவல்களை சரி பார்த்து உங்களுக்கு உடனே சொல்றேன். [short pause] கொஞ்சம் நேரம் காத்திருங்க, இணைப்பு இப்போ தான் தொடங்குது.")
    ]
    for lang,text in tests:
        result=requests.post("http://127.0.0.1:8091/debug/tts",
                             headers=HEADERS,json={"text":text,"language":lang},timeout=300)
        result.raise_for_status()
        print("Language:",lang,"| Characters sent:",len(text),
              "| Characters accepted:",result.headers.get("X-TTS-Input-Characters"),
              "| Speech chunks:",result.headers.get("X-TTS-Chunks"))
        assert int(result.headers["X-TTS-Input-Characters"]) == len(text), "Input was truncated!"
        target=Path("/kaggle/working") / ("jarvis_long_"+lang+".wav")
        target.write_bytes(result.content)
        print("WAV duration:",round(sf.info(target).duration,2),"seconds")
        display(Audio(filename=str(target)))
    print("LONG-FORM TEST DONE. Verify the final words are audible.")
else:
    print("Long test skipped. Change RUN_LONG_TTS_TEST=True when you want to verify complete paragraphs.")

## 10. Create ONE temporary authenticated WebSocket tunnel

Use this only after **both** debug tests pass. Cloudflare Quick Tunnel URLs are temporary and publicly reachable; the server separately checks the private Authorization token. Do not share your secret.

In [ ]:
import re
CLOUDFLARED = Path("/kaggle/working/cloudflared")
if not CLOUDFLARED.is_file() or CLOUDFLARED.stat().st_size < 100000:
    subprocess.run(["wget", "-q",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "-O", str(CLOUDFLARED)], check=True)
CLOUDFLARED.chmod(0o755)

if "tunnel" in globals() and tunnel is not None and tunnel.poll() is None:
    tunnel.terminate()
    tunnel.wait(timeout=15)

TUNNEL_LOG = Path("/kaggle/working/cloudflared.log")
tunnel_log_handle = TUNNEL_LOG.open("w")
tunnel = subprocess.Popen(
    [str(CLOUDFLARED), "tunnel", "--url", "http://127.0.0.1:8091"],
    stdout=tunnel_log_handle, stderr=subprocess.STDOUT
)
WS_URL = None
for _ in range(60):
    if tunnel.poll() is not None:
        break
    time.sleep(1)
    matches = re.findall(
        r"https://[a-z0-9-]+\.trycloudflare\.com",
        TUNNEL_LOG.read_text(errors="replace")
    )
    if matches:
        WS_URL = matches[-1].replace("https://", "wss://") + "/ws"
        break
if not WS_URL:
    print(TUNNEL_LOG.read_text(errors="replace")[-2000:])
    raise RuntimeError("Tunnel did not start. Check Kaggle Internet permissions.")
print("VOICE_WS_URL =", WS_URL)
print("PASS: use this URL in Windows .env and the SAME rotated token from Kaggle Secrets")

## 11. Windows PowerShell — run on your PC, NOT in Kaggle

Use headphones during the first test. In **PowerShell**:

```powershell
cd C:\Users\LENOVO\jarvis-realtime-voice
git pull --ff-only
.\.venv\Scripts\python.exe -m pip install -r requirements-duplex-client.txt

# First verify your Windows mic and speaker work simultaneously:
.\.venv\Scripts\python.exe local_audio_smoke.py

# Update only connection settings:
notepad .env
```

Edit your existing Windows `.env` (do not put this in GitHub):

```ini
VOICE_WS_URL=wss://THE-URL-FROM-STEP-10/ws
VOICE_API_TOKEN=YOUR-NEW-KAGGLE-SECRET-VALUE
AEC_ENABLED=0
VAD_MODE=2
```

Then start:

```powershell
.\.venv\Scripts\python.exe duplex_client.py
```

Speak **once**, then wait. Expect `[transcript]`, `[stage:llm]`, streamed reply text, `[stage:tts]`, and `[metric] first_pcm_after_commit_ms`, followed by audible speech. **The initial configuration intentionally disables partial-ASR; that is normal.**

If Kaggle restarted after Step 10, the tunnel URL may have changed. Update `.env` accordingly.

## 12. Read server logs if anything fails (optional)

This cell does not restart anything. Only run it if your Windows output has an error or silence.

In [ ]:
print("Server running:", server.poll() is None)
print("Tunnel running:", "tunnel" in globals() and tunnel.poll() is None)
print("Server log (last 4500 characters):")
print(SERVER_LOG.read_text(errors="replace")[-4500:])

## 13. After first spoken reply: measure and gradually re-enable partial ASR/Smart Turn

Your previous latency metric measured time from server turn commit to the first PCM chunk, **not** the whole user-speaking-to-hearing delay. The fallback endpoint pause (now 850 ms by default), WebSocket round trip and Windows audio playback add time.

Once the speech pipeline works reliably:
1. On Windows, press Ctrl+C.
2. In Step 5 set `PARTIAL_ASR_ENABLED="1"` if you want interim recognition, and restart the server (rerun Step 7). Check whether GPU0 stays stable.
3. Optionally set `SMART_TURN_ENABLED="1"` to evaluate prosody-based pauses; its Tamil/Tanglish accuracy needs real testing.
4. Try `ASR_LANGUAGE="en"` for an English-only session or `"ta"` for Tamil-only to avoid auto-language errors; for mixed bilingual sessions use `"auto"`.
5. If English voice is fine but Tamil is too slow, try `OMNIVOICE_STEPS="8"` then `"12"` and compare quality/timing.
6. Use headphones during interruption tests to prevent speaker audio feeding back into ASR.

**Unresolved engineering work:** true native streaming TTS and semantic overlapping-speech understanding. This notebook switches the temporary brain and reduces errors/latency; it cannot guarantee human-level real-time speech.